In [0]:
# Parametros. Edita los valores antes de ejecutar el notebook.
dbutils.widgets.text('catalogo', 'electrocasa')
dbutils.widgets.text('container', 'electrocasa')
dbutils.widgets.text('landing_subpath', 'landing')

catalogo = dbutils.widgets.get('catalogo')
container = dbutils.widgets.get('container')
landing_subpath = dbutils.widgets.get('landing_subpath').strip('/')

storage_account = "sagianellacelis"
sql_host = 'analyticsdmc.database.windows.net'
sql_database = 'electrocasadb'

print(f'Catalogo: {catalogo}')
print(f'Landing: abfss://{container}@{storage_account}.dfs.core.windows.net/{landing_subpath}')


In [0]:
spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalogo}')
for schema in ['bronze', 'silver', 'gold', 'observability']:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalogo}.{schema}')


In [0]:
volume_url = f'abfss://{container}@{storage_account}.dfs.core.windows.net/{landing_subpath}'
spark.sql(f"""
CREATE EXTERNAL VOLUME IF NOT EXISTS {catalogo}.bronze.landing
LOCATION '{volume_url}'
""")


In [0]:
base = f'/Volumes/{catalogo}/bronze/landing'
carpetas = [
    'ventas', 'productos', 'empleados', 'resenas', 'devoluciones',
    'schemas/ventas', 'schemas/empleados', 'schemas/resenas', 'schemas/devoluciones',
    'config'
]
for carpeta in carpetas:
    dbutils.fs.mkdirs(f'{base}/{carpeta}')
print('Estructura landing creada')


In [0]:
# GRANT/REVOKE 
for grupo in ['electrocasa_engineers', 'electrocasa_analysts', 'electrocasa_auditors']:
    spark.sql(f'GRANT USE CATALOG ON CATALOG {catalogo} TO `{grupo}`')

for schema in ['bronze', 'silver', 'gold', 'observability']:
    spark.sql(f'GRANT USE SCHEMA ON SCHEMA {catalogo}.{schema} TO `electrocasa_engineers`')
    spark.sql(f'GRANT SELECT, MODIFY, CREATE TABLE, CREATE FUNCTION ON SCHEMA {catalogo}.{schema} TO `electrocasa_engineers`')

spark.sql(f'GRANT USE SCHEMA ON SCHEMA {catalogo}.gold TO `electrocasa_analysts`')
spark.sql(f'GRANT SELECT ON SCHEMA {catalogo}.gold TO `electrocasa_analysts`')

for schema in ['gold', 'observability']:
    spark.sql(f'GRANT USE SCHEMA ON SCHEMA {catalogo}.{schema} TO `electrocasa_auditors`')
    spark.sql(f'GRANT SELECT ON SCHEMA {catalogo}.{schema} TO `electrocasa_auditors`')

# analysts/auditors no deben leer Bronze ni Silver.
#for grupo in ['electrocasa_analysts', 'electrocasa_auditors']:
#    for schema in ['bronze', 'silver']:
#        spark.sql(f'REVOKE USE SCHEMA ON SCHEMA {catalogo}.{schema} FROM `{grupo}`')
#        spark.sql(f'REVOKE SELECT ON SCHEMA {catalogo}.{schema} FROM `{grupo}`')


## Conexion a SQL Server


In [0]:
spark.sql(f"""
CREATE CONNECTION IF NOT EXISTS electrocasa_sql
TYPE SQLSERVER
OPTIONS (
  host '{sql_host}',
  port '1433',
  user secret('electrocasa-secrets', 'user'),
  password secret('electrocasa-secrets', 'password')
)
""")

spark.sql(f"""
CREATE FOREIGN CATALOG IF NOT EXISTS electrocasa_sql_catalog
USING CONNECTION electrocasa_sql
OPTIONS (database '{sql_database}')
""")


In [0]:
# Validaciones finales sin revelar secretos.
display(spark.sql(f'SHOW SCHEMAS IN {catalogo}'))
display(spark.sql('SHOW CONNECTIONS'))
